# DH-FDM-PINN plate bending under SINUSOIDAL loading

In [1]:
"""
DH-FDM-PINN (Deterministic-Harmonic Fourier-feature input embedding +
Finite-Difference-Method PINN backbone) for Kirchhoff-Love plate bending
under SINUSOIDAL loading, all-simply-supported (SS-SS-SS-SS).

This is the DH variant of fdm_pinn_sinusoidal_hardBC.py. Everything about
the physics setup, hard-constrained BC, derivative computation (FD
stencils on a ghost-extended grid), training loop, output artifacts, and
plotting is UNCHANGED. The ONLY thing that changes is the network's INPUT
BLOCK: instead of feeding the raw normalized coordinates (x/a, y/b)
straight into the hidden MLP, we first lift them into a small deterministic
harmonic (DH) Fourier-feature embedding.

-----------------------------------------------------------------------------
WHAT "DETERMINISTIC-HARMONIC" MEANS HERE (and how it differs from RFF)
-----------------------------------------------------------------------------
"Random Fourier Features" (RFF), the usual PINN spectral-bias fix, project
the input through a FIXED-BUT-RANDOM Gaussian matrix B ~ N(0, sigma^2):
  gamma(x) = [sin(2*pi*B*x), cos(2*pi*B*x)]
This helps with spectral bias but (a) introduces a sigma hyperparameter
that must be tuned per problem, and (b) makes the embedding itself
stochastic / run-dependent.

DH-FDM-PINN instead uses a FIXED, DETERMINISTIC, INTEGER harmonic basis
tied to the physical domain and the mathematical structure of the exact
solution family:
    gamma(x, y) = [ xi, eta,
                     sin(k*pi*xi), cos(k*pi*xi),
                     sin(k*pi*eta), cos(k*pi*eta)   for k = 1..K_MODES ]
  where xi = x/a, eta = y/b.

Rationale: the Navier series solution of a simply-supported rectangular
plate is itself a double sine series, w(x,y) = sum_{m,n} A_mn
sin(m*pi*x/a) sin(n*pi*y/b). Seeding the network's input embedding with
exactly this harmonic family gives the MLP a head start representing
members of that family (including their derivatives, which is what the
FD stencils differentiate), with:
  - No sigma / bandwidth hyperparameter to tune (RFF's main pain point).
  - No randomness in the embedding itself -- gamma(x,y) is a pure
    deterministic function of (x,y) and K_MODES, independent of SEED.
    (SEED still governs the MLP weight initialization as before.)
  - Frequencies chosen to match the problem's natural length scales (a, b)
    rather than sampled from an arbitrary distribution.

This is still a "Fourier feature" input embedding in the general PINN
sense (it lifts low-dimensional coordinates into a higher-dimensional
space of sinusoids before the MLP, combating spectral bias / helping the
network fit high-frequency and boundary-adjacent content) -- it is just
DETERMINISTIC and HARMONIC (integer multiples of the domain's fundamental
frequency) rather than random.

GHOST-POINT SAFETY: gamma(x,y) is a smooth, everywhere-defined function of
xi=x/a, eta=y/b (plain sin/cos), so it evaluates fine on the ghost-extended
grid (xi, eta outside [0,1]) with no discontinuities or special-casing --
exactly like the hard-BC envelope already did in the FDM-PINN parent
script. No changes needed anywhere in the FD stencil machinery.

SCOPE (identical to fdm_pinn_sinusoidal_hardBC.py otherwise):
  - pde_loss (biharmonic residual) and moment_loss (Mx=0/My=0 on edges)
    -> computed via FD stencils on one ghost-extended forward pass, as
    before. Only the forward pass itself now runs through the DH
    embedding first.
  - Hard-constrained essential BC (w=0 on all 4 edges via the envelope
    D(x,y) = (x/a)(1-x/a)(y/b)(1-y/b)) -- UNCHANGED.
  - Post-processing / plotting -- UNCHANGED.

CAVEATS:
  - K_MODES controls embedding width (2 + 4*K_MODES input features to the
    first Linear layer). Larger K_MODES = richer harmonic basis but more
    parameters in the first layer and a small extra forward-pass cost;
    K_MODES=4 is a reasonable default for a single-sine-mode-dominated
    load case like this one. Sweep it if under/over-fitting is suspected.
  - This embedding is a strong inductive bias FOR problems whose solution
    is well-approximated by a low-order double sine series (Navier-type
    SS-SS-SS-SS problems, exactly this benchmark). It is not a
    general-purpose spectral-bias fix for arbitrary BCs/geometries the
    way RFF is; do not assume it transfers as-is to e.g. clamped edges or
    non-rectangular domains without reconsidering the basis.
  - Everything else about accuracy/cost trade-offs from the FDM-PINN
    parent docstring (FD truncation error ~O(h^4), ghost-grid forward-pass
    overhead, fixed-uniform-grid requirement) still applies unchanged.

Outputs (all written to ./results/<MODEL_NAME>/) -- SAME schema as the
vanilla/FDM scripts, so downstream comparison/aggregation scripts keep
working:
  - best_weights.pt, training_convergence.csv, cost_metrics.json,
    error_metrics.json, loss_balance.json, field_data.csv, figures/*.png

Run:
    python dh_fdm_pinn_sinusoidal_hardBC.py
"""

import os
import json
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import cm

# ============================================================
# CONFIG
# ============================================================
MODEL_NAME = "DH_FDM_PINN_HardBC"   # >>> VARIANT POINT: change per architecture
LOADING_CASE = "sinusoidal"

OUT_DIR = os.path.join("results", MODEL_NAME)
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

SEED = 42
MAX_EPOCHS = 15000
PATIENCE = 3000
PRINT_EVERY = 100
LR_INIT = 1e-3
LR_DECAY_STEPS = 2000
LR_DECAY_RATE = 0.95

# --- Deterministic-Harmonic (DH) Fourier-feature embedding config --------
K_MODES = 4  # number of integer harmonics per axis (k = 1..K_MODES)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

# ============================================================
# SEEDING
# ============================================================
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================================
# PHYSICAL / NON-DIMENSIONAL PARAMETERS
# ============================================================
real_a, real_b, real_h = 1.0, 1.0, 0.01
real_E, real_nu, real_q0 = 200e9, 0.3, 1000.0

D_dim = real_E * (real_h ** 3) / (12 * (1 - real_nu ** 2))
L = real_a
W = real_q0 * (L ** 4) / D_dim
D_nd = 1.0
a_nd, b_nd, q0_nd, nu_nd = 1.0, 1.0, 1.0, real_nu

# ============================================================
# DOMAIN
# ============================================================
nx, ny = 51, 51
x_nd = np.linspace(0, a_nd, nx)
y_nd = np.linspace(0, b_nd, ny)
X_nd, Y_nd = np.meshgrid(x_nd, y_nd)   # 'xy' indexing -> shape (ny, nx)
grid_points_nd = np.vstack([X_nd.ravel(), Y_nd.ravel()]).T

dx = a_nd / (nx - 1)
dy = b_nd / (ny - 1)

center_x, center_y = a_nd / 2, b_nd / 2
center_idx = int(np.argmin((grid_points_nd[:, 0] - center_x) ** 2 +
                            (grid_points_nd[:, 1] - center_y) ** 2))

# --- Full boundary -- kept only for the diagnostic bc_loss now ------------
boundary_indices = np.unique(np.concatenate([
    np.where(grid_points_nd[:, 1] == 0)[0],
    np.where(grid_points_nd[:, 1] == b_nd)[0],
    np.where(grid_points_nd[:, 0] == 0)[0],
    np.where(grid_points_nd[:, 0] == a_nd)[0],
]))
interior_indices = np.setdiff1d(np.arange(nx * ny), boundary_indices)

bc_idx = torch.tensor(boundary_indices, dtype=torch.long, device=DEVICE)
int_idx = torch.tensor(interior_indices, dtype=torch.long, device=DEVICE)

# --- per-edge indices for the moment BCs ------------------------------
edge_x0 = np.where(grid_points_nd[:, 0] == 0)[0]
edge_xa = np.where(grid_points_nd[:, 0] == a_nd)[0]
edge_y0 = np.where(grid_points_nd[:, 1] == 0)[0]
edge_yb = np.where(grid_points_nd[:, 1] == b_nd)[0]

edge_x_indices = np.unique(np.concatenate([edge_x0, edge_xa]))  # Mx = 0 here
edge_y_indices = np.unique(np.concatenate([edge_y0, edge_yb]))  # My = 0 here

edge_x_idx = torch.tensor(edge_x_indices, dtype=torch.long, device=DEVICE)
edge_y_idx = torch.tensor(edge_y_indices, dtype=torch.long, device=DEVICE)

# ============================================================
# GHOST-EXTENDED GRID (for finite-difference stencils)
# ============================================================
# GHOST=3 is the minimum needed by the 4th-order-accurate 4th-derivative
# stencil (offsets -3..+3). Do not reduce this.
GHOST = 3

x_ext_1d = (np.arange(-GHOST, nx + GHOST)) * dx   # length nx + 2*GHOST
y_ext_1d = (np.arange(-GHOST, ny + GHOST)) * dy   # length ny + 2*GHOST
X_ext, Y_ext = np.meshgrid(x_ext_1d, y_ext_1d)    # shape (ny+2G, nx+2G)

X_ext_flat = torch.tensor(X_ext.reshape(-1, 1), dtype=DTYPE, device=DEVICE)
Y_ext_flat = torch.tensor(Y_ext.reshape(-1, 1), dtype=DTYPE, device=DEVICE)
EXT_NY = ny + 2 * GHOST
EXT_NX = nx + 2 * GHOST

# Physical (non-dimensional) grid, as a torch tensor, for evaluating the load
X_nd_t = torch.tensor(X_nd, dtype=DTYPE, device=DEVICE)
Y_nd_t = torch.tensor(Y_nd, dtype=DTYPE, device=DEVICE)

# --- 4th-order-accurate central finite-difference coefficients (Fornberg) --
# 1st derivative, 5-pt, offsets -2..2
C1 = torch.tensor([1 / 12, -8 / 12, 0.0, 8 / 12, -1 / 12], dtype=DTYPE, device=DEVICE)
# 2nd derivative, 5-pt, offsets -2..2
C2 = torch.tensor([-1 / 12, 16 / 12, -30 / 12, 16 / 12, -1 / 12], dtype=DTYPE, device=DEVICE)
# 4th derivative, 7-pt, offsets -3..3
C4 = torch.tensor([-1 / 6, 2.0, -13 / 2, 28 / 3, -13 / 2, 2.0, -1 / 6], dtype=DTYPE, device=DEVICE)


def _apply_stencil_axis(arr, coeffs, axis, ghost, out_len):
    """Apply a centered 1D stencil (`coeffs`, offsets -half..+half) along
    `axis` of `arr`, where `arr` has `ghost` extra points on each side of
    that axis beyond the first `out_len` physical points. Returns a tensor
    with `out_len` along `axis` and all other dimensions unchanged."""
    half = (coeffs.shape[0] - 1) // 2
    assert ghost >= half, "not enough ghost padding for this stencil"
    out = 0.0
    for k in range(-half, half + 1):
        c = coeffs[k + half]
        start = ghost + k
        if axis == 0:
            sl = arr[start:start + out_len, ...]
        else:
            sl = arr[:, start:start + out_len]
        out = out + c * sl
    return out


def d1_axis(arr, axis, ghost, out_len, h):
    return _apply_stencil_axis(arr, C1, axis, ghost, out_len) / h


def d2_axis(arr, axis, ghost, out_len, h):
    return _apply_stencil_axis(arr, C2, axis, ghost, out_len) / (h ** 2)


def d4_axis(arr, axis, ghost, out_len, h):
    return _apply_stencil_axis(arr, C4, axis, ghost, out_len) / (h ** 4)


def fd_derivatives(model):
    """Single forward pass on the ghost-extended grid, then every
    derivative needed for training and post-processing as finite
    differences of that one pass. All returned tensors have shape
    (ny, nx), matching X_nd/Y_nd's row=y, col=x convention."""
    w_ext = model(X_ext_flat, Y_ext_flat).reshape(EXT_NY, EXT_NX)

    # Pure-x derivatives: physical y rows, full x ghost range
    w_rows = w_ext[GHOST:GHOST + ny, :]                              # (ny, nx+2G)
    w_xxxx = d4_axis(w_rows, axis=1, ghost=GHOST, out_len=nx, h=dx)   # (ny, nx)
    w_xx = d2_axis(w_rows, axis=1, ghost=GHOST, out_len=nx, h=dx)     # (ny, nx)

    # Pure-y derivatives: full y ghost range, physical x cols
    w_cols = w_ext[:, GHOST:GHOST + nx]                               # (ny+2G, nx)
    w_yyyy = d4_axis(w_cols, axis=0, ghost=GHOST, out_len=ny, h=dy)   # (ny, nx)
    w_yy = d2_axis(w_cols, axis=0, ghost=GHOST, out_len=ny, h=dy)     # (ny, nx)

    # Mixed: w_xxyy = d2/dx2 ( d2/dy2 w ), applied as two sequential 1D stencils
    w_yy_full = d2_axis(w_ext, axis=0, ghost=GHOST, out_len=ny, h=dy)  # (ny, nx+2G)
    w_xxyy = d2_axis(w_yy_full, axis=1, ghost=GHOST, out_len=nx, h=dx)  # (ny, nx)

    # Mixed: w_xy = d/dx ( d/dy w )
    w_y_full = d1_axis(w_ext, axis=0, ghost=GHOST, out_len=ny, h=dy)   # (ny, nx+2G)
    w_xy = d1_axis(w_y_full, axis=1, ghost=GHOST, out_len=nx, h=dx)    # (ny, nx)

    w_phys = w_ext[GHOST:GHOST + ny, GHOST:GHOST + nx]                 # (ny, nx)

    return w_phys, w_xx, w_yy, w_xxxx, w_yyyy, w_xxyy, w_xy


# ============================================================
# ANALYTICAL SOLUTIONS (Navier series, sinusoidal load, SS-SS-SS-SS)
# ============================================================
def analytical_w_nd(x, y):
    denom = np.pi ** 4 * D_nd * (1 / a_nd ** 2 + 1 / b_nd ** 2) ** 2
    return (q0_nd / denom) * np.sin(np.pi * x / a_nd) * np.sin(np.pi * y / b_nd)


def analytical_Mx_nd(x, y):
    denom = np.pi ** 2 * (1 / a_nd ** 2 + 1 / b_nd ** 2) ** 2
    return (q0_nd / denom) * (1 / a_nd ** 2 + nu_nd / b_nd ** 2) * np.sin(np.pi * x / a_nd) * np.sin(np.pi * y / b_nd)


def analytical_My_nd(x, y):
    denom = np.pi ** 2 * (1 / a_nd ** 2 + 1 / b_nd ** 2) ** 2
    return (q0_nd / denom) * (nu_nd / a_nd ** 2 + 1 / b_nd ** 2) * np.sin(np.pi * x / a_nd) * np.sin(np.pi * y / b_nd)


def analytical_Mxy_nd(x, y):
    denom = ((1 / a_nd ** 2) + (1 / b_nd ** 2)) ** 2
    return (-q0_nd * (1 - nu_nd) / (a_nd * b_nd * np.pi ** 2 * denom)) * np.cos(np.pi * x / a_nd) * np.cos(np.pi * y / b_nd)


def sinusoidal_load_nd(x, y):
    return q0_nd * torch.sin(np.pi * x / a_nd) * torch.sin(np.pi * y / b_nd)


Q_grid = sinusoidal_load_nd(X_nd_t, Y_nd_t)   # (ny, nx), precomputed once


# ============================================================
# DETERMINISTIC-HARMONIC (DH) FOURIER-FEATURE INPUT EMBEDDING
# ============================================================
def dh_fourier_features(x, y):
    """Deterministic harmonic embedding of (x, y):
        [xi, eta,
         sin(k*pi*xi), cos(k*pi*xi), sin(k*pi*eta), cos(k*pi*eta)]_{k=1..K_MODES}
    where xi = x/a_nd, eta = y/b_nd. Output width = 2 + 4*K_MODES.

    Fixed integer frequencies (no sigma hyperparameter, no randomness) --
    matches the harmonic family of the Navier series analytical solution
    for this SS-SS-SS-SS problem. Pure elementwise sin/cos of a smooth
    linear function of (x, y), so it is safe to evaluate on ghost points
    outside [0, a] x [0, b] with no special-casing.
    """
    xi = x / a_nd
    eta = y / b_nd
    feats = [xi, eta]
    for k in range(1, K_MODES + 1):
        kpi = k * np.pi
        feats.append(torch.sin(kpi * xi))
        feats.append(torch.cos(kpi * xi))
        feats.append(torch.sin(kpi * eta))
        feats.append(torch.cos(kpi * eta))
    return torch.cat(feats, dim=1)


DH_EMBED_DIM = 2 + 4 * K_MODES


# ============================================================
# MODEL   (>>> VARIANT POINT: DH Fourier-feature input block, FDM backbone)
# ============================================================
class PlatePINN(nn.Module):
    def __init__(self, hidden=64, n_layers=3):
        super().__init__()
        layers = []
        in_dim = DH_EMBED_DIM   # was 2 in the plain/FDM scripts; now the DH embedding width
        for _ in range(n_layers):
            layers.append(nn.Linear(in_dim, hidden, dtype=DTYPE))
            layers.append(nn.SiLU())
            in_dim = hidden
        self.hidden = nn.Sequential(*layers)
        self.out = nn.Linear(in_dim, 1, dtype=DTYPE)
        self.scale = nn.Parameter(torch.tensor(1.0, dtype=DTYPE))

    def forward(self, x, y):
        h = dh_fourier_features(x, y)   # >>> DH Fourier-feature embedding (was raw [x/a, y/b] cat)
        h = self.hidden(h)
        raw = self.out(h)

        # Hard-constrained essential BC (w = 0 on all edges), unchanged from
        # the vanilla/FDM hard-BC scripts. Evaluated fine for ghost points
        # outside [0,a]x[0,b] too -- smooth polynomial continuation.
        xi = x / a_nd
        eta = y / b_nd
        envelope = xi * (1 - xi) * eta * (1 - eta)
        return self.scale * envelope * raw


model = PlatePINN().to(DEVICE)

# ============================================================
# OPTIMIZER + LR SCHEDULE
# ============================================================
optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lambda step: LR_DECAY_RATE ** (step / LR_DECAY_STEPS)
)


# ============================================================
# LOSS (4th-order derivatives via finite differences, not autograd)
# ============================================================
def compute_loss():
    w_grid, w_xx, w_yy, w_xxxx, w_yyyy, w_xxyy, w_xy = fd_derivatives(model)

    biharmonic = w_xxxx + 2 * w_xxyy + w_yyyy
    residual = (biharmonic - Q_grid).reshape(-1, 1)
    pde_loss = torch.mean(residual[int_idx] ** 2)

    w_flat = w_grid.reshape(-1, 1)
    # DIAGNOSTIC ONLY -- should be ~0 exactly, by construction of the model
    # (hard-constrained envelope). Not included in `total`.
    bc_loss = torch.mean(w_flat[bc_idx] ** 2)

    Mx = -(w_xx + nu_nd * w_yy)
    My = -(w_yy + nu_nd * w_xx)
    Mx_flat = Mx.reshape(-1, 1)
    My_flat = My.reshape(-1, 1)
    moment_loss = torch.mean(Mx_flat[edge_x_idx] ** 2) + torch.mean(My_flat[edge_y_idx] ** 2)

    total = pde_loss + moment_loss
    return total, pde_loss, bc_loss, moment_loss, w_flat


# ============================================================
# TRAINING LOOP with computational-cost logging + early stopping
# ============================================================
epoch_history, loss_history, pde_history, bc_history, moment_history = [], [], [], [], []
w_center_history, epoch_time_history = [], []

best_loss = float("inf")
wait = 0
weights_path = os.path.join(OUT_DIR, "best_weights.pt")

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

train_start = time.perf_counter()
converged_epoch = MAX_EPOCHS

for epoch in range(MAX_EPOCHS):
    epoch_t0 = time.perf_counter()

    optimizer.zero_grad()
    total, pde, bc, moment, w_pred = compute_loss()
    total.backward()
    optimizer.step()
    scheduler.step()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    epoch_dt = time.perf_counter() - epoch_t0

    loss_val = total.item()
    w_center_val = w_pred[center_idx, 0].item()

    epoch_history.append(epoch)
    loss_history.append(loss_val)
    pde_history.append(pde.item())
    bc_history.append(bc.item())
    moment_history.append(moment.item())
    w_center_history.append(w_center_val)
    epoch_time_history.append(epoch_dt)

    if epoch % PRINT_EVERY == 0:
        print(f"Epoch {epoch} Loss {loss_val:.6e} (pde {pde.item():.3e}, "
              f"moment {moment.item():.3e}, hardBC_residual {bc.item():.3e}) "
              f"Center {w_center_val:.6e} ({epoch_dt*1000:.2f} ms/epoch)")

    if loss_val < best_loss:
        best_loss = loss_val
        wait = 0
        torch.save(model.state_dict(), weights_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch}")
            converged_epoch = epoch
            break

total_train_time = time.perf_counter() - train_start
peak_mem_mb = (torch.cuda.max_memory_allocated() / 1e6) if DEVICE.type == "cuda" else float("nan")

model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
model.eval()

# ============================================================
# SAVE TRAINING CONVERGENCE CSV
# ============================================================
w_center_dim_history = np.array(w_center_history) * W
df_hist = pd.DataFrame({
    "epoch": epoch_history,
    "total_loss": loss_history,
    "pde_loss": pde_history,
    "hard_bc_residual_diagnostic": bc_history,
    "moment_loss": moment_history,
    "w_center_nd": w_center_history,
    "w_center_dim": w_center_dim_history,
    "epoch_time_s": epoch_time_history,
})
df_hist.to_csv(os.path.join(OUT_DIR, "training_convergence.csv"), index=False)
print("Saved training_convergence.csv")

# ============================================================
# LOSS BALANCE DIAGNOSTIC
# ============================================================
final_pde, final_bc, final_moment = pde_history[-1], bc_history[-1], moment_history[-1]
final_total = final_pde + final_moment  # bc excluded, it's ~0 by construction
loss_balance = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "final_pde_loss": final_pde,
    "final_moment_loss": final_moment,
    "final_hard_bc_residual_diagnostic": final_bc,
    "pde_fraction_of_total": final_pde / final_total,
    "moment_fraction_of_total": final_moment / final_total,
}
dominant_fraction = max(
    loss_balance["pde_fraction_of_total"],
    loss_balance["moment_fraction_of_total"],
)
if dominant_fraction > 0.9:
    print(f"WARNING: one loss term is {dominant_fraction*100:.1f}% of the total "
          f"unweighted loss -- consider a weighting scheme before trusting these results.")
if final_bc > 1e-20:
    print(f"WARNING: hard_bc_residual_diagnostic = {final_bc:.3e} is not ~0 -- "
          f"double-check the envelope function / grid point exactness (float64 expected).")
with open(os.path.join(OUT_DIR, "loss_balance.json"), "w") as f:
    json.dump(loss_balance, f, indent=2)
print("Saved loss_balance.json:", loss_balance)

# ============================================================
# COMPUTATIONAL COST METRICS
# ============================================================
cost_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "device": str(DEVICE),
    "epochs_run": len(epoch_history),
    "epochs_to_early_stop": converged_epoch,
    "epochs_to_convergence": converged_epoch,  # deprecated alias, same value
    "mean_time_per_epoch_s": float(np.mean(epoch_time_history)),
    "median_time_per_epoch_s": float(np.median(epoch_time_history)),
    "total_training_time_s": total_train_time,
    "peak_gpu_memory_mb": peak_mem_mb,
    "n_trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "ghost_points_per_side": GHOST,
    "forward_pass_points": EXT_NX * EXT_NY,
    "dh_k_modes_per_axis": K_MODES,
    "dh_embedding_dim": DH_EMBED_DIM,
}
with open(os.path.join(OUT_DIR, "cost_metrics.json"), "w") as f:
    json.dump(cost_metrics, f, indent=2)
print("Saved cost_metrics.json:", cost_metrics)

# ============================================================
# POST-PROCESSING: full-field predictions (w, Mx, My, Mxy) via the
# same finite-difference machinery used in training.
# ============================================================
def compute_full_fields():
    with torch.no_grad():
        w_grid, w_xx, w_yy, _, _, _, w_xy = fd_derivatives(model)
        Mx = -(w_xx + nu_nd * w_yy)
        My = -(w_yy + nu_nd * w_xx)
        Mxy = -(1 - nu_nd) * w_xy

    w_grid_np = w_grid.cpu().numpy()
    Mx_np = Mx.cpu().numpy()
    My_np = My.cpu().numpy()
    Mxy_np = Mxy.cpu().numpy()
    return w_grid_np, Mx_np, My_np, Mxy_np


w_pred_nd, Mx_pred_nd, My_pred_nd, Mxy_pred_nd = compute_full_fields()

w_anal_nd = analytical_w_nd(X_nd, Y_nd)
Mx_anal_nd = analytical_Mx_nd(X_nd, Y_nd)
My_anal_nd = analytical_My_nd(X_nd, Y_nd)
Mxy_anal_nd = analytical_Mxy_nd(X_nd, Y_nd)

w_dim_pred, w_dim_anal = w_pred_nd * W, w_anal_nd * W
moment_scale = D_dim * W / L ** 2
Mx_dim_pred, Mx_dim_anal = Mx_pred_nd * moment_scale, Mx_anal_nd * moment_scale
My_dim_pred, My_dim_anal = My_pred_nd * moment_scale, My_anal_nd * moment_scale
Mxy_dim_pred, Mxy_dim_anal = Mxy_pred_nd * moment_scale, Mxy_anal_nd * moment_scale

w_signed_error = w_dim_pred - w_dim_anal
w_error = np.abs(w_signed_error)
Mx_error = np.abs(Mx_dim_pred - Mx_dim_anal)
My_error = np.abs(My_dim_pred - My_dim_anal)
Mxy_error = np.abs(Mxy_dim_pred - Mxy_dim_anal)

# ============================================================
# ERROR METRICS
# ============================================================
ix, iy = nx // 2, ny // 2
w_pred_center, w_anal_center = w_dim_pred[iy, ix], w_dim_anal[iy, ix]
center_abs_err = abs(w_pred_center - w_anal_center)
center_pct_err = center_abs_err / abs(w_anal_center) * 100

L2_error_w = np.sqrt(np.mean((w_dim_pred - w_dim_anal) ** 2))
rel_L2_error_w = np.linalg.norm(w_dim_pred - w_dim_anal) / np.linalg.norm(w_dim_anal)

error_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "center_deflection_pred_m": float(w_pred_center),
    "center_deflection_anal_m": float(w_anal_center),
    "center_abs_error_m": float(center_abs_err),
    "center_pct_error": float(center_pct_err),
    "w_max_abs_error_m": float(np.max(w_error)),
    "w_rms_error_m": float(np.sqrt(np.mean(w_error ** 2))),
    "w_L2_error_m": float(L2_error_w),
    "w_relative_L2_error": float(rel_L2_error_w),
    "Mx_max_abs_error_Nm": float(np.max(Mx_error)),
    "Mx_rms_error_Nm": float(np.sqrt(np.mean(Mx_error ** 2))),
    "My_max_abs_error_Nm": float(np.max(My_error)),
    "My_rms_error_Nm": float(np.sqrt(np.mean(My_error ** 2))),
    "Mxy_max_abs_error_Nm": float(np.max(Mxy_error)),
    "Mxy_rms_error_Nm": float(np.sqrt(np.mean(Mxy_error ** 2))),
    "boundary_w_max_abs_value_m": float(np.max(np.abs(w_dim_pred[0, :]))
                                         + np.max(np.abs(w_dim_pred[-1, :]))
                                         + np.max(np.abs(w_dim_pred[:, 0]))
                                         + np.max(np.abs(w_dim_pred[:, -1]))),
}
with open(os.path.join(OUT_DIR, "error_metrics.json"), "w") as f:
    json.dump(error_metrics, f, indent=2)
print("Saved error_metrics.json:", error_metrics)

df_field = pd.DataFrame({
    "x_nd": X_nd.ravel(), "y_nd": Y_nd.ravel(),
    "w_dim_anal": w_dim_anal.ravel(), "w_dim_pred": w_dim_pred.ravel(),
    "Mx_dim_anal": Mx_dim_anal.ravel(), "Mx_dim_pred": Mx_dim_pred.ravel(),
    "My_dim_anal": My_dim_anal.ravel(), "My_dim_pred": My_dim_pred.ravel(),
    "Mxy_dim_anal": Mxy_dim_anal.ravel(), "Mxy_dim_pred": Mxy_dim_pred.ravel(),
})
df_field["w_abs_error"] = np.abs(df_field["w_dim_anal"] - df_field["w_dim_pred"])
df_field["w_rel_error"] = df_field["w_abs_error"] / (np.abs(df_field["w_dim_anal"]) + 1e-12)
df_field.to_csv(os.path.join(OUT_DIR, "field_data.csv"), index=False)
print("Saved field_data.csv")

# ============================================================
# PLOTTING
# ============================================================
plt.rcParams.update({"figure.dpi": 150})


def save_contour(field, title, cbar_label, fname, cmap=cm.jet):
    plt.figure(figsize=(7, 6))
    cf = plt.contourf(X_nd, Y_nd, field, levels=50, cmap=cmap)
    plt.colorbar(cf, label=cbar_label)
    plt.title(title, fontstyle="italic", fontsize=13)
    plt.xlabel("x/a", fontstyle="italic")
    plt.ylabel("y/b", fontstyle="italic")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, fname))
    plt.close()


plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], df_hist["w_center_dim"], linewidth=2, label=f"{MODEL_NAME} Prediction")
plt.axhline(y=w_anal_center, linestyle="--", linewidth=2, color="k", label="Analytical")
plt.xlabel("Epoch"); plt.ylabel("Center Deflection w (m)")
plt.title("Convergence of Center Deflection")
plt.legend(); plt.grid(True, linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Center_Deflection.png")); plt.close()

plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], np.abs(df_hist["w_center_dim"] - w_anal_center), linewidth=2)
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("Absolute Error at Center (m)")
plt.title("Convergence Error (Log Scale)")
plt.grid(True, which="both", linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Error_Log_Scale.png")); plt.close()

plt.figure(figsize=(8, 6))
plt.plot(df_hist["epoch"], df_hist["total_loss"], label="Total Loss")
plt.plot(df_hist["epoch"], df_hist["pde_loss"], label="PDE Loss", alpha=0.7)
plt.plot(df_hist["epoch"], df_hist["moment_loss"], label="Moment Loss", alpha=0.7)
plt.plot(df_hist["epoch"], df_hist["hard_bc_residual_diagnostic"],
         label="Hard-BC residual (diagnostic, ~0)", alpha=0.5, linestyle=":")
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training Loss Curve")
plt.legend(); plt.grid(True, which="both", ls="--", alpha=0.7); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Training_Loss_Curve.png")); plt.close()

save_contour(w_dim_anal, "Analytical Plate Deflection", "Deflection (m)",
             "Analytical_Plate_Deflection.png")
save_contour(w_dim_pred, f"{MODEL_NAME} Predicted Plate Deflection", "Deflection (m)",
             "PINN_Predicted_Plate_Deflection.png")
save_contour(w_error, "Absolute Error in Plate Deflection", "|w_pred - w_anal| (m)",
             "Absolute_Error_in_Plate_Deflection.png", cmap=cm.inferno)
save_contour(w_signed_error, "Signed Error w_pred - w_anal", "w_pred - w_anal (m)",
             "Absolute_Error_w_pred_-_w_anal.png", cmap="coolwarm")

save_contour(Mx_dim_anal, "Analytical Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x_Analytical.png")
save_contour(Mx_dim_pred, "Predicted Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x.png")
save_contour(Mx_error, "Absolute Error in $M_x$", "|Mx_pred - Mx_anal| (N·m/m)",
             "Absolute_Error_in_M_x.png", cmap="hot")

save_contour(My_dim_anal, "Analytical Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y_Analytical.png")
save_contour(My_dim_pred, "Predicted Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y.png")
save_contour(My_error, "Absolute Error in $M_y$", "|My_pred - My_anal| (N·m/m)",
             "Absolute_Error_in_M_y.png", cmap="hot")

save_contour(Mxy_dim_anal, "Analytical Twisting Moment $M_{xy}$", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy_Analytical.png")
save_contour(Mxy_dim_pred, "Predicted Twisting Moment $M_{xy}$", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy.png")
save_contour(Mxy_error, "Absolute Error in $M_{xy}$", "|Mxy_pred - Mxy_anal| (N·m/m)",
             "Absolute_Error_in_M_xy.png", cmap="hot")

plt.figure(figsize=(6, 6))
plt.scatter(w_dim_anal.ravel(), w_dim_pred.ravel(), s=8, alpha=0.5)
lims = [min(w_dim_anal.min(), w_dim_pred.min()), max(w_dim_anal.max(), w_dim_pred.max())]
plt.plot(lims, lims, "k--", linewidth=1, label="Perfect agreement")
plt.xlabel("Analytical displacement w (m)", fontstyle="italic")
plt.ylabel("Predicted displacement w (m)", fontstyle="italic")
plt.title("Analytical vs Predicted Displacement", fontstyle="italic")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Analytical_vs_Predicted_Displacement.png")); plt.close()

err_fft = np.fft.fftshift(np.fft.fft2(w_error))
err_fft_mag = np.log1p(np.abs(err_fft))
plt.figure(figsize=(7, 6))
plt.imshow(err_fft_mag, cmap="magma", extent=[-nx // 2, nx // 2, -ny // 2, ny // 2])
plt.colorbar(label="log(1 + |FFT error|)")
plt.title(f"Error Spectrum (FFT) — {LOADING_CASE}")
plt.xlabel("kx"); plt.ylabel("ky")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Error_Spectrum_FFT.png")); plt.close()

fig, axes = plt.subplots(4, 3, figsize=(16, 18))
rows = [
    ("Displacement w (m)", w_dim_anal, w_dim_pred, w_error),
    ("Moment Mx (N·m)", Mx_dim_anal, Mx_dim_pred, Mx_error),
    ("Moment My (N·m)", My_dim_anal, My_dim_pred, My_error),
    ("Moment Mxy (N·m)", Mxy_dim_anal, Mxy_dim_pred, Mxy_error),
]
for r, (label, anal, pred, err) in enumerate(rows):
    for c, (title, field, cmap) in enumerate([
        (f"Analytical {label}", anal, cm.jet),
        (f"Predicted {label}", pred, cm.jet),
        (f"Absolute Error {label}", err, cm.inferno),
    ]):
        ax = axes[r, c]
        cf = ax.contourf(X_nd, Y_nd, field, levels=20, cmap=cmap)
        fig.colorbar(cf, ax=ax)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("x/a"); ax.set_ylabel("y/b")
plt.suptitle(f"{MODEL_NAME} — {LOADING_CASE} loading, SS-SS-SS-SS (summary grid)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "_summary_grid_all_fields.png")); plt.close()

print(f"\nAll outputs written to: {os.path.abspath(OUT_DIR)}")
print(f"Total training time: {total_train_time:.2f}s | "
      f"Epochs to early-stop: {converged_epoch} | "
      f"Peak GPU mem: {peak_mem_mb:.1f} MB" if DEVICE.type == "cuda" else
      f"\nAll outputs written to: {os.path.abspath(OUT_DIR)} | "
      f"Total training time: {total_train_time:.2f}s | Epochs: {converged_epoch}")

Epoch 0 Loss 6.243138e+01 (pde 6.243e+01, moment 4.651e-03, hardBC_residual 0.000e+00) Center -6.363122e-03 (357.30 ms/epoch)
Epoch 100 Loss 1.014929e-01 (pde 1.014e-01, moment 1.138e-04, hardBC_residual 0.000e+00) Center 2.311049e-03 (11.12 ms/epoch)
Epoch 200 Loss 2.899902e-02 (pde 2.839e-02, moment 6.078e-04, hardBC_residual 0.000e+00) Center 3.868873e-03 (11.10 ms/epoch)
Epoch 300 Loss 1.616799e-02 (pde 1.543e-02, moment 7.361e-04, hardBC_residual 0.000e+00) Center 4.074184e-03 (11.96 ms/epoch)
Epoch 400 Loss 1.090553e-02 (pde 1.012e-02, moment 7.865e-04, hardBC_residual 0.000e+00) Center 4.149369e-03 (12.28 ms/epoch)
Epoch 500 Loss 8.085216e-03 (pde 7.279e-03, moment 8.058e-04, hardBC_residual 0.000e+00) Center 4.182404e-03 (12.37 ms/epoch)
Epoch 600 Loss 6.341837e-03 (pde 5.532e-03, moment 8.101e-04, hardBC_residual 0.000e+00) Center 4.195364e-03 (12.42 ms/epoch)
Epoch 700 Loss 5.166224e-03 (pde 4.359e-03, moment 8.069e-04, hardBC_residual 0.000e+00) Center 4.197865e-03 (12.46 ms

C:\Users\vivek\AppData\Local\Temp\ipykernel_6064\3797378465.py:466: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path, map_location

Saved training_convergence.csv
Saved loss_balance.json: {'model': 'DH_FDM_PINN_HardBC', 'loading_case': 'sinusoidal', 'final_pde_loss': 0.00019923166973672148, 'final_moment_loss': 7.932478361884726e-06, 'final_hard_bc_residual_diagnostic': 0.0, 'pde_fraction_of_total': 0.9617092125510587, 'moment_fraction_of_total': 0.038290787448941294}
Saved cost_metrics.json: {'model': 'DH_FDM_PINN_HardBC', 'loading_case': 'sinusoidal', 'device': 'cuda', 'epochs_run': 15000, 'epochs_to_early_stop': 15000, 'epochs_to_convergence': 15000, 'mean_time_per_epoch_s': 0.011154424853040837, 'median_time_per_epoch_s': 0.01082824997138232, 'total_training_time_s': 179.4892356999917, 'peak_gpu_memory_mb': 30.386176, 'n_trainable_params': 9602, 'ghost_points_per_side': 3, 'forward_pass_points': 3249, 'dh_k_modes_per_axis': 4, 'dh_embedding_dim': 18}
Saved error_metrics.json: {'model': 'DH_FDM_PINN_HardBC', 'loading_case': 'sinusoidal', 'center_deflection_pred_m': 0.00014678196922693378, 'center_deflection_anal